In [1]:
# Decorations
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pprint import pprint
import gmplot
from config import api_key
from pprint import pprint

In [2]:
# Import database
df1 = pd.read_csv('Data/housing.csv')
df2 = pd.read_csv('Data/Total.csv')
df1 = df1.dropna()

### df1(1990) Analysis

Sort and slice data to get top and bottom 5%

In [ ]:
# SSort data by house value
df1 = df1.sort_values('median_house_value', ascending=False).reset_index(drop=True)
df1.head()

In [ ]:
# Slice the top 5%
top5 = df1.iloc[:round(len(df1['median_house_value'])*0.05), :].sort_values('median_house_value', ascending=True)

In [ ]:
# slice the bottom 5%
bottom5 = df1.iloc[round(len(df1['median_house_value'])*0.95): len(df1['median_house_value']),:].sort_values('median_house_value', ascending=True)

Plotting Top 5% and Bottom 5% on gmap (Use gmplot package)

In [ ]:
# Set the center of the map
gmap = gmplot.GoogleMapPlotter(top5['latitude'].median(),
                                   top5['longitude'].median(), 100000)
# Plot scatter points based on LatLng
gmap.scatter(top5['latitude'], top5['longitude'], '#FF0000', 
                              size = 2000, marker = False ) 
gmap.scatter(bottom5['latitude'], bottom5['longitude'], '#110870', 
                              size = 2000, marker = False ) 
# Draw out to 'Plot' folder in html format
gmap.draw("Plot/gmap.html")

### df2(2018) Analysis 

In [3]:
# Keep only the columns that are useful 
df2 = df2[['Sub Type', 'St#', 'St Name', 'City', 'L/C Price', 'Br/Ba', 'YrBuilt']]
df2.head()
#df2 = df2.drop(on=0)

,Sub Type,St#,St Name,City,L/C Price,Br/Ba,YrBuilt
0,MANL/D,12,Via Veneto #12,CHU,"$89,000","2/2,0,0,0",2006
1,CONDO/A,1653,Pentecost Way #1,SD,"$158,000","1/1,0,0,0",1980
2,CONDO/A,6101,Adelaide Ave #107,SD,"$168,250","0/1,0,0,0",1984/ASR
3,CONDO/A,589 N,Johnson #137,ELC,"$175,000","2/1,0,0,0",1980
4,SFR,3138,Roadrunner,BSPG,"$190,000","2/2,0,0,0",1986


In [4]:
yr_blt = df2.loc[:, 'YrBuilt'].str.split('/', expand=True)[0]
yr_blt = pd.DataFrame(yr_blt)
yr_blt = yr_blt.fillna(0)

In [5]:
df2['Age'] = ''
count = 0
for i in yr_blt[0]:
        i = 2018 - int(i)
        df2['Age'][count] = i
        count += 1
df2.head()

,Sub Type,St#,St Name,City,L/C Price,Br/Ba,YrBuilt,Age
0,MANL/D,12,Via Veneto #12,CHU,"$89,000","2/2,0,0,0",2006,12
1,CONDO/A,1653,Pentecost Way #1,SD,"$158,000","1/1,0,0,0",1980,38
2,CONDO/A,6101,Adelaide Ave #107,SD,"$168,250","0/1,0,0,0",1984/ASR,34
3,CONDO/A,589 N,Johnson #137,ELC,"$175,000","2/1,0,0,0",1980,38
4,SFR,3138,Roadrunner,BSPG,"$190,000","2/2,0,0,0",1986,32


In [6]:
df2['house_price'] = df2.loc[:, 'L/C Price'].str.split('$', expand=True)[1] 

In [7]:
df2.head()

,Sub Type,St#,St Name,City,L/C Price,Br/Ba,YrBuilt,Age,house_price
0,MANL/D,12,Via Veneto #12,CHU,"$89,000","2/2,0,0,0",2006,12,"89,000"
1,CONDO/A,1653,Pentecost Way #1,SD,"$158,000","1/1,0,0,0",1980,38,"158,000"
2,CONDO/A,6101,Adelaide Ave #107,SD,"$168,250","0/1,0,0,0",1984/ASR,34,"168,250"
3,CONDO/A,589 N,Johnson #137,ELC,"$175,000","2/1,0,0,0",1980,38,"175,000"
4,SFR,3138,Roadrunner,BSPG,"$190,000","2/2,0,0,0",1986,32,"190,000"


In [8]:
df2_by_value = df2.set_index("L/C Price")
df2_by_value = df2_by_value.reset_index()
df2_by_value.head()

,L/C Price,Sub Type,St#,St Name,City,Br/Ba,YrBuilt,Age,house_price
0,"$89,000",MANL/D,12,Via Veneto #12,CHU,"2/2,0,0,0",2006,12,"89,000"
1,"$158,000",CONDO/A,1653,Pentecost Way #1,SD,"1/1,0,0,0",1980,38,"158,000"
2,"$168,250",CONDO/A,6101,Adelaide Ave #107,SD,"0/1,0,0,0",1984/ASR,34,"168,250"
3,"$175,000",CONDO/A,589 N,Johnson #137,ELC,"2/1,0,0,0",1980,38,"175,000"
4,"$190,000",SFR,3138,Roadrunner,BSPG,"2/2,0,0,0",1986,32,"190,000"


### Getting LatLng using geocoder API for df2

In [18]:
# import CityCode
citycode_df = pd.read_csv('Data/CleanCityCode.csv')
citycode_df = citycode_df.iloc[1:, [1,2]].reset_index(drop=True)
citycode_df.head()

,City,Code
0,29 Palms,29P
1,Acton,AC
2,Adelanto,AD
3,Agoura,AGO
4,Agoura Hills,AGHI


In [24]:
df2_by_value['City name'] = ""
for index_df2, row_df2 in df2_by_value.iterrows():
    for index, row in citycode_df.iterrows():
        if row_df2['City'] == row['Code']:
            row_df2['City name'] = row['City']
            print(row_df2['City name'])

CHU
29P   
CHU
AC     
CHU
AD    
CHU
AGO   
CHU
AGHI 
CHU
ADUL 
CHU
AGNA  
CHU
ALHL 
CHU
ALH   
CHU
AV    
CHU
ALP 
CHU
ALTL  
CHU
ALTA  
CHU
ANA   
CHU
AH    
CHU
ANGO  
CHU
ANZA  
CHU
APPV  
CHU
ARCD  
CHU
ARL    
CHU
AB 
CHU
ART   
CHU
ATW 
CHU
AVAL 
CHU
AZU   
CHU
BKSF 
CHU
BDPK  
CHU
BANN  
CHU
BSTW  
CHU
BST 
CHU
BMT   
CHU
BELA 
CHU
BELL  
CHU
BC     
CHU
BG    
CHU
BF    
CHU
BD    
CHU
BEVH 
CHU
BB    
CHU
BLM   
CHU
BLY 
CHU
BON 
CHU
BNSL 
CHU
BSPG 
CHU
BOU 
CHU
BRAD  
CHU
BREA  
CHU
BREN 
CHU
BP    
CHU
BULH 
CHU
BBK 
CHU
CABA  
CHU
CPS    
CHU
CLB   
CHU
CBH   
CHU
CBP    
CHU
CACY  
CHU
CALI  
CHU
CAM    
CHU
CAMP 
CHU
CP    
CHU
CA    
CHU
CANC 
CHU
CLK   
CHU
CAS 
CHU
CAR 
CHU
CAV 
CHU
CARS 
CHU
CAST 
CHU
CC 
CHU
CER   
CHU
CHT   
CHU
CHVL  
CHU
CH    
CHU
CHH   
CHU
CHU 
CHU
COM   
CHU
COI 
CHU
CLM 
CHU
CLAR  
CHU
COA 
CHU
CLTN  
CHU
CMP   
CHU
COR   
CHU
CDM   
CHU
CORO 
CHU
CM    
CHU
CDC   
CHU
COTT 
CHU
CVN   
CHU
CRLN  
CHU
CUD   
CHU
CULV  
CHU
CYP   
CHU
DP    


DES 
SD
DCTR 
SD
DHS 
SD
DS 
SD
DEVO 
SD
DB    
SD
DOW   
SD
DUAR 
SD
DUL 
SD
EAGL 
SD
ESD 
SD
ELC 
SD
ELCE 
SD
ELM   
SD
ES    
SD
ESER 
SD
ENCS 
SD
ENC 
SD
ESC 
SD
FBRK 
SD
FIL    
SD
FONT  
SD
FC 
SD
FRFL 
SD
FV    
SD
FRZ    
SD
FRVL 
SD
FUL   
SD
GG    
SD
GR 
SD
GRVL 
SD
GARN 
SD
GPRK 
SD
GAV 
SD
GLHL 
SD
GLIV 
SD
GD    
SD
GLDR 
SD
GDHP 
SD
GH    
SD
GRTR  
SD
GRV    
SD
GRVC 
SD
GUA 
SD
HH    
SD
HC    
SD
HVCY 
SD
HG    
SD
HAWT 
SD
HMT   
SD
HMB 
SD
HSP   
SD
HHIL 
SD
HGRV 
SD
HLND  
SD
HDPK 
SD
HLDL 
SD
HLW 
SD
HOME 
SD
HNLU 
SD
HNBK 
SD
HB    
SD
HNPK  
SD
IWLD  
SD
IMB 
SD
IMC 
SD
IW 
SD
IND 
SD
ING 
SD
IR    
SD
IR 
SD
IRW 
SD
JAC 
SD
JAM 
SD
JT 
SD
JUL 
SD
JFLT 
SD
KGY    
SD
LCF   
SD
LACR 
SD
LH    
SD
LHH   
SD
LAJ 
SD
LMSA 
SD
LAM   
SD
LP    
SD
LPUE  
SD
LQ 
SD
LS 
SD
LVRN  
SD
LADH 
SD
LB    
SD
LAGH  
SD
LN    
SD
LAGW  
SD
LKAR  
SD
EZB   
SD
LKEL 
SD
LKEL  
SD
LF    
SD
LF 
SD
LHSU 
SD
LKH    
SD
LKL    
SD
LKMA  
SD
LKS    
SD
LAK 
SD
LVT    
SD
LW    
SD
LNCR

BSPG
GD    
BSPG
GLDR 
BSPG
GDHP 
BSPG
GH    
BSPG
GRTR  
BSPG
GRV    
BSPG
GRVC 
BSPG
GUA 
BSPG
HH    
BSPG
HC    
BSPG
HVCY 
BSPG
HG    
BSPG
HAWT 
BSPG
HMT   
BSPG
HMB 
BSPG
HSP   
BSPG
HHIL 
BSPG
HGRV 
BSPG
HLND  
BSPG
HDPK 
BSPG
HLDL 
BSPG
HLW 
BSPG
HOME 
BSPG
HNLU 
BSPG
HNBK 
BSPG
HB    
BSPG
HNPK  
BSPG
IWLD  
BSPG
IMB 
BSPG
IMC 
BSPG
IW 
BSPG
IND 
BSPG
ING 
BSPG
IR    
BSPG
IR 
BSPG
IRW 
BSPG
JAC 
BSPG
JAM 
BSPG
JT 
BSPG
JUL 
BSPG
JFLT 
BSPG
KGY    
BSPG
LCF   
BSPG
LACR 
BSPG
LH    
BSPG
LHH   
BSPG
LAJ 
BSPG
LMSA 
BSPG
LAM   
BSPG
LP    
BSPG
LPUE  
BSPG
LQ 
BSPG
LS 
BSPG
LVRN  
BSPG
LADH 
BSPG
LB    
BSPG
LAGH  
BSPG
LN    
BSPG
LAGW  
BSPG
LKAR  
BSPG
EZB   
BSPG
LKEL 
BSPG
LKEL  
BSPG
LF    
BSPG
LF 
BSPG
LHSU 
BSPG
LKH    
BSPG
LKL    
BSPG
LKMA  
BSPG
LKS    
BSPG
LAK 
BSPG
LVT    
BSPG
LW    
BSPG
LNCR 
BSPG
LNDR 
BSPG
LAWN 
BSPG
LEG 
BSPG
LX 
BSPG
LVY 
BSPG
LHTS 
BSPG
LIV 
BSPG
LTLR 
BSPG
LOH 
BSPG
LOML 
BSPG
LOM 
BSPG
LONG  
BSPG
LOSA 
BSPG
LA  
BSPG
CO    
BSPG
LFE 


LMSA
LAWN 
LMSA
LEG 
LMSA
LX 
LMSA
LVY 
LMSA
LHTS 
LMSA
LIV 
LMSA
LTLR 
LMSA
LOH 
LMSA
LOML 
LMSA
LOM 
LMSA
LONG  
LMSA
LOSA 
LMSA
LA  
LMSA
CO    
LMSA
LFE 
LMSA
LUCV 
LMSA
LNWD 
LMSA
LCRK 
LMSA
MLBU 
LMSA
MANH 
LMSA
MARV 
LMSA
MR 
LMSA
MW    
LMSA
MVLY 
LMSA
MBRK 
LMSA
MEC 
LMSA
MENF  
LMSA
MENT  
LMSA
MC    
LMSA
MILO  
LMSA
MIM 
LMSA
MIR 
LMSA
MIH 
LMSA
MH    
LMSA
MIV 
LMSA
MV    
LMSA
MNRO  
LMSA
MCLR  
LMSA
MTB   
LMSA
MHTS 
LMSA
MHIL 
LMSA
MP    
LMSA
MTR 
LMSA
MPK    
LMSA
MPK   
LMSA
MORV  
LMSA
MGOV 
LMSA
MRBY 
LMSA
MTBL  
LMSA
MOL 
LMSA
MWSH 
LMSA
MNTC 
LMSA
MUR 
LMSA
MUR   
LMSA
NAC 
LMSA
NEED 
LMSA
NBS 
LMSA
NPK   
LMSA
NEWH 
LMSA
NB    
LMSA
NC    
LMSA
NORC  
LMSA
NOH    
LMSA
NHLW  
LMSA
NOP 
LMSA
NSH 
LMSA
NTUS 
LMSA
NTUS  
LMSA
NR    
LMSA
NWK   
LMSA
NUEV  
LMSA
OAKG 
LMSA
OAS 
LMSA
OCB 
LMSA
OCE 
LMSA
OJAI   
LMSA
OLAN 
LMSA
OTS 
LMSA
ONT   
LMSA
ORG   
LMSA
ORG 
LMSA
OPA 
LMSA
ORTG 
LMSA
OTM 
LMSA
OA 
LMSA
OS 
LMSA
OXN    
LMSA
PAB 
LMSA
PACP 
LMSA
PAC    
LMSA
PA

MDRA
RDOM  
MDRA
RM    
MDRA
RPV 
MDRA
RPV  
MDRA
REDL  
MDRA
RDMN 
MDRA
REDO 
MDRA
RES    
MDRA
RLT   
MDRA
RCR 
MDRA
RVSD  
MDRA
RH  
MDRA
RHE 
MDRA
RMLD  
MDRA
RMNA 
MDRA
ROSA 
MDRA
RSMD  
MDRA
ROSS  
MDRA
ROW 
MDRA
RBDX 
MDRA
RSA 
MDRA
SAGE 
MDRA
SALT 
MDRA
SB    
MDRA
SAC 
MDRA
SC    
MDRA
SD 
MDRA
SDMS  
MDRA
SF     
MDRA
SGAB  
MDRA
SJCN  
MDRA
SJOS 
MDRA
SJ    
MDRA
SLR 
MDRA
SAM 
MDRA
SMRO  
MDRA
SP 
MDRA
SYSI 
MDRA
SCN    
MDRA
SA 
MDRA
SA    
MDRA
SAH 
MDRA
SCL 
MDRA
SFS   
MDRA
SMCA 
MDRA
SPL    
MDRA
SAR    
MDRA
SYSA 
MDRA
SAN 
MDRA
SAU    
MDRA
SCM 
MDRA
SLB   
MDRA
SEDC 
MDRA
SEQM 
MDRA
SHH   
MDRA
SO    
MDRA
SMAD  
MDRA
SIGH  
MDRA
SVL 
MDRA
SIM   
MDRA
SIM    
MDRA
SV 
MDRA
SOL 
MDRA
SOM    
MDRA
SOR 
MDRA
SELM  
MDRA
SOG   
MDRA
SPAS  
MDRA
SPV 
MDRA
STAN  
MDRA
SR     
MDRA
STUD 
MDRA
SGLF 
MDRA
SUNC  
MDRA
SUNV 
MDRA
SUNL  
MDRA
SNMD 
MDRA
SSB   
MDRA
SURF  
MDRA
SYL   
MDRA
TAR   
MDRA
TEC 
MDRA
THP    
MDRA
TEM 
MDRA
TEM   
MDRA
TMPL  
MDRA
TH 
MDRA
TO    
MDRA


SAN 
JMST
SAU    
JMST
SCM 
JMST
SLB   
JMST
SEDC 
JMST
SEQM 
JMST
SHH   
JMST
SO    
JMST
SMAD  
JMST
SIGH  
JMST
SVL 
JMST
SIM   
JMST
SIM    
JMST
SV 
JMST
SOL 
JMST
SOM    
JMST
SOR 
JMST
SELM  
JMST
SOG   
JMST
SPAS  
JMST
SPV 
JMST
STAN  
JMST
SR     
JMST
STUD 
JMST
SGLF 
JMST
SUNC  
JMST
SUNV 
JMST
SUNL  
JMST
SNMD 
JMST
SSB   
JMST
SURF  
JMST
SYL   
JMST
TAR   
JMST
TEC 
JMST
THP    
JMST
TEM 
JMST
TEM   
JMST
TMPL  
JMST
TH 
JMST
TO    
JMST
TO     
JMST
TP 
JMST
TIE 
JMST
TUL    
JMST
TOP    
JMST
TORR 
JMST
TUJ   
JMST
TUS 
JMST
TUS   
JMST
UCTY 
JMST
UNIV 
JMST
UPL   
JMST
VAL   
JMST
VAL    
JMST
VC 
JMST
VG    
JMST
VVL   
JMST
VNE    
JMST
VNW    
JMST
VEN 
JMST
VENT 
JMST
VERN 
JMST
VTVL  
JMST
VCVL 
JMST
VP    
JMST
VST   
JMST
WAL   
JMST
WP    
JMST
WSPR  Anza/Aguanga/Warner Springs 
JMST
WCOV  West Covina 
JMST
WEH    
JMST
WHO 
JMST
WS  
JMST
WLV   
JMST
WTM   
JMST
WH    
JMST
WDVL 
JMST
WILD  
JMST
WILM 
JMST
WVLY 
JMST
WINC  
JMST
WIN    
JMST
WOOD 
JMST
WHLL 

... (372 outputs left out)

SJOS
LF 
SJOS
LHSU 
SJOS
LKH    
SJOS
LKL    
SJOS
LKMA  
SJOS
LKS    
SJOS
LAK 
SJOS
LVT    
SJOS
LW    
SJOS
LNCR 
SJOS
LNDR 
SJOS
LAWN 
SJOS
LEG 
SJOS
LX 
SJOS
LVY 
SJOS
LHTS 
SJOS
LIV 
SJOS
LTLR 
SJOS
LOH 
SJOS
LOML 
SJOS
LOM 
SJOS
LONG  
SJOS
LOSA 
SJOS
LA  
SJOS
CO    
SJOS
LFE 
SJOS
LUCV 
SJOS
LNWD 
SJOS
LCRK 
SJOS
MLBU 
SJOS
MANH 
SJOS
MARV 
SJOS
MR 
SJOS
MW    
SJOS
MVLY 
SJOS
MBRK 
SJOS
MEC 
SJOS
MENF  
SJOS
MENT  
SJOS
MC    
SJOS
MILO  
SJOS
MIM 
SJOS
MIR 
SJOS
MIH 
SJOS
MH    
SJOS
MIV 
SJOS
MV    
SJOS
MNRO  
SJOS
MCLR  
SJOS
MTB   
SJOS
MHTS 
SJOS
MHIL 
SJOS
MP    
SJOS
MTR 
SJOS
MPK    
SJOS
MPK   
SJOS
MORV  
SJOS
MGOV 
SJOS
MRBY 
SJOS
MTBL  
SJOS
MOL 
SJOS
MWSH 
SJOS
MNTC 
SJOS
MUR 
SJOS
MUR   
SJOS
NAC 
SJOS
NEED 
SJOS
NBS 
SJOS
NPK   
SJOS
NEWH 
SJOS
NB    
SJOS
NC    
SJOS
NORC  
SJOS
NOH    
SJOS
NHLW  
SJOS
NOP 
SJOS
NSH 
SJOS
NTUS 
SJOS
NTUS  
SJOS
NR    
SJOS
NWK   
SJOS
NUEV  
SJOS
OAKG 
SJOS
OAS 
SJOS
OCB 
SJOS
OCE 
SJOS
OJAI   
SJOS
OLAN 
SJOS
OTS 
SJOS
ONT 

MTVW
29P   
MTVW
AC     
MTVW
AD    
MTVW
AGO   
MTVW
AGHI 
MTVW
ADUL 
MTVW
AGNA  
MTVW
ALHL 
MTVW
ALH   
MTVW
AV    
MTVW
ALP 
MTVW
ALTL  
MTVW
ALTA  
MTVW
ANA   
MTVW
AH    
MTVW
ANGO  
MTVW
ANZA  
MTVW
APPV  
MTVW
ARCD  
MTVW
ARL    
MTVW
AB 
MTVW
ART   
MTVW
ATW 
MTVW
AVAL 
MTVW
AZU   
MTVW
BKSF 
MTVW
BDPK  
MTVW
BANN  
MTVW
BSTW  
MTVW
BST 
MTVW
BMT   
MTVW
BELA 
MTVW
BELL  
MTVW
BC     
MTVW
BG    
MTVW
BF    
MTVW
BD    
MTVW
BEVH 
MTVW
BB    
MTVW
BLM   
MTVW
BLY 
MTVW
BON 
MTVW
BNSL 
MTVW
BSPG 
MTVW
BOU 
MTVW
BRAD  
MTVW
BREA  
MTVW
BREN 
MTVW
BP    
MTVW
BULH 
MTVW
BBK 
MTVW
CABA  
MTVW
CPS    
MTVW
CLB   
MTVW
CBH   
MTVW
CBP    
MTVW
CACY  
MTVW
CALI  
MTVW
CAM    
MTVW
CAMP 
MTVW
CP    
MTVW
CA    
MTVW
CANC 
MTVW
CLK   
MTVW
CAS 
MTVW
CAR 
MTVW
CAV 
MTVW
CARS 
MTVW
CAST 
MTVW
CC 
MTVW
CER   
MTVW
CHT   
MTVW
CHVL  
MTVW
CH    
MTVW
CHH   
MTVW
CHU 
MTVW
COM   
MTVW
COI 
MTVW
CLM 
MTVW
CLAR  
MTVW
COA 
MTVW
CLTN  
MTVW
CMP   
MTVW
COR   
MTVW
CDM   
MTVW
CORO 
MTVW
CM    


CABA  
APTO
CPS    
APTO
CLB   
APTO
CBH   
APTO
CBP    
APTO
CACY  
APTO
CALI  
APTO
CAM    
APTO
CAMP 
APTO
CP    
APTO
CA    
APTO
CANC 
APTO
CLK   
APTO
CAS 
APTO
CAR 
APTO
CAV 
APTO
CARS 
APTO
CAST 
APTO
CC 
APTO
CER   
APTO
CHT   
APTO
CHVL  
APTO
CH    
APTO
CHH   
APTO
CHU 
APTO
COM   
APTO
COI 
APTO
CLM 
APTO
CLAR  
APTO
COA 
APTO
CLTN  
APTO
CMP   
APTO
COR   
APTO
CDM   
APTO
CORO 
APTO
CM    
APTO
CDC   
APTO
COTT 
APTO
CVN   
APTO
CRLN  
APTO
CUD   
APTO
CULV  
APTO
CYP   
APTO
DP    
APTO
DP 
APTO
DEC 
APTO
DEM 
APTO
DES 
APTO
DCTR 
APTO
DHS 
APTO
DS 
APTO
DEVO 
APTO
DB    
APTO
DOW   
APTO
DUAR 
APTO
DUL 
APTO
EAGL 
APTO
ESD 
APTO
ELC 
APTO
ELCE 
APTO
ELM   
APTO
ES    
APTO
ESER 
APTO
ENCS 
APTO
ENC 
APTO
ESC 
APTO
FBRK 
APTO
FIL    
APTO
FONT  
APTO
FC 
APTO
FRFL 
APTO
FV    
APTO
FRZ    
APTO
FRVL 
APTO
FUL   
APTO
GG    
APTO
GR 
APTO
GRVL 
APTO
GARN 
APTO
GPRK 
APTO
GAV 
APTO
GLHL 
APTO
GLIV 
APTO
GD    
APTO
GLDR 
APTO
GDHP 
APTO
GH    
APTO
GRTR  
APTO
GRV    
APT

KeyboardInterrupt: 

In [ ]:
df2_by_value["Address"] = df2_by_value["St#"].astype(str) + " " + df2_by_value["St Name"].astype(str) + " " + df2_by_value['City']
df2_by_value['Lat'] = ""
df2_by_value['Lng'] = ""

In [ ]:
params = {"key": api_key}
for index, row in df2_by_value.iterrows():
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params['address'] = row['Address']
    geo_data = requests.get(base_url, params).json()
    try:
        df2_by_value["Lat"][index] = geo_data["results"][0]["geometry"]["location"]["lat"]
        df2_by_value["Lng"][index] = geo_data["results"][0]["geometry"]["location"]["lng"]
    except IndexError:
        print(f"Row {index} cannot be found on gmap.")
        continue

In [ ]:
# Output to CSV
df2_by_value.to_csv("Data/df2_latlng.csv")

### Plotting df2_latlng

In [ ]:
df2_latlng = pd.read_csv("Data/df2_latlng.csv")

In [ ]:
df2_latlng

In [ ]:
# Set the center of the map
gmap_df2 = gmplot.GoogleMapPlotter(df2_latlng['Lat'].median(),
                                df2_latlng['Lng'].median(), 100000)
# Plot scatter points based on LatLng
gmap_df2.scatter(df2_latlng['Lat'], df2_latlng['Lng'], '#FF0000', 
                              size = 10000, marker = False) 
# Draw out to 'Plot' folder in html format
gmap_df2.draw("Plot/gmap_df2.html")

## df1 vs df2